# 09 · Full Pipeline Demo

End-to-end: render or load an image -> ML prediction -> constellation
overlay -> world-map pin (assuming the field center is at the
observer's zenith and using a chosen UTC timestamp).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np

from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field, sample_random_pointing
from src.inference.predict import load_model, predict_image
from src.inference.visualize import plot_constellation_overlay, plot_world_map
from astropy.time import Time

In [ ]:
model = load_model(ROOT / 'checkpoints/best.pt', device='cpu')
catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=6.0)
rng = np.random.default_rng(11)

In [ ]:
for case in range(3):
    ra, dec, rot, fw = sample_random_pointing(rng, (20.0, 60.0))
    img = render_star_field(ra, dec, fw, rot, catalog, image_size=224, rng=np.random.default_rng(case))
    img_u8 = (img * 255).astype(np.uint8)
    pred = predict_image(model, img_u8)

    # Sky overlay using the model's prediction (not the truth).
    fig1 = plot_constellation_overlay(img_u8,
        ra_center=pred.ra_deg, dec_center=pred.dec_deg,
        field_width_deg=pred.field_width_deg, rotation_deg=pred.rotation_deg,
        catalog=catalog, mag_limit=5.5,
        title=f'Pred RA={pred.ra_deg:.1f}° Dec={pred.dec_deg:.1f}° (truth {ra:.1f}, {dec:.1f})')
    fig1.savefig(ROOT / f'reports/figures/09_demo_overlay_{case}.png', dpi=140); plt.show()

    # Convert to a tentative lat/lon assuming zenith pointing at a fixed UTC.
    when_utc = datetime(2026, 1, 15, 22, 0, 0, tzinfo=timezone.utc)
    gmst_deg = Time(when_utc).sidereal_time('mean', 'greenwich').hour * 15.0
    lat = float(np.clip(pred.dec_deg, -90, 90))
    lon = ((pred.ra_deg - gmst_deg + 540.0) % 360.0) - 180.0
    fig2 = plot_world_map(lat, lon, title=f'Case {case}: implied observer lat/lon')
    fig2.savefig(ROOT / f'reports/figures/09_demo_map_{case}.png', dpi=140); plt.show()